Find the total number of available beds per hosts' nationality.
Output the nationality along with the corresponding total number of available beds. Sort records by the total available beds in descending order.

In [0]:
%skip
CREATE TABLE ska_catalog.bronze.airbnb_apartments(host_id int,apartment_id varchar(5),apartment_type varchar(10),n_beds int,n_bedrooms int,country varchar(20),city varchar(20));
INSERT INTO ska_catalog.bronze.airbnb_apartments VALUES(0,'A1','Room',1,1,'USA','NewYork'),(0,'A2','Room',1,1,'USA','NewJersey'),(0,'A3','Room',1,1,'USA','NewJersey'),(1,'A4','Apartment',2,1,'USA','Houston'),(1,'A5','Apartment',2,1,'USA','LasVegas'),(3,'A7','Penthouse',3,3,'China','Tianjin'),(3,'A8','Penthouse',5,5,'China','Beijing'),(4,'A9','Apartment',2,1,'Mali','Bamako'),(5,'A10','Room',3,1,'Mali','Segou');

CREATE TABLE ska_catalog.bronze.airbnb_hosts(host_id int,nationality  varchar(15),gender varchar(5),age int);
INSERT INTO ska_catalog.bronze.airbnb_hosts  VALUES(0,'USA','M',28),(1,'USA','F',29),(2,'China','F',31),(3,'China','M',24),(4,'Mali','M',30),(5,'Mali','F',30);

In [0]:
SELECT * FROM ska_catalog.bronze.airbnb_hosts;

In [0]:
SELECT * FROM ska_catalog.bronze.airbnb_apartments;

In [0]:
%python

'''
SELECT h.nationality AS `Nationality`, SUM(n_beds) AS `No_of_Beds` FROM ska_catalog.bronze.airbnb_apartments a
JOIN ska_catalog.bronze.airbnb_hosts h
ON a.host_id = h.host_id
GROUP BY h.nationality
ORDER BY No_of_Beds DESC;
'''
from pyspark.sql.functions import *
# ska_catalog.bronze.airbnb_apartments
df_apart = spark.sql("SELECT * FROM ska_catalog.bronze.airbnb_apartments")
# ska_catalog.bronze.airbnb_hosts
df_host = spark.sql("SELECT * FROM ska_catalog.bronze.airbnb_hosts")

df_result = df_apart.join(df_host, df_apart.host_id == df_host.host_id) \
    .groupBy(df_host.nationality.alias("Nationality")) \
    .agg(sum("n_beds").alias("No_of_Beds")) \
    .orderBy(col("No_of_Beds").desc())

display(df_result)

In [0]:
%python
"""SELECT apartment_id, apartment_type, country 
FROM ska_catalog.bronze.airbnb_apartments
WHERE country LIKE '%USA%'
ORDER BY apartment_id ASC;"""

df_usa = df_apart.select('apartment_id', 'apartment_type', 'country') \
    .where(col('country').like('%USA%')) \
    .orderBy(col('apartment_id'))

display(df_usa)


In [0]:
%python
"""SELECT host_id AS `HOST_ID`, COUNT(apartment_id) AS `APARTMENT_CNT`
FROM ska_catalog.bronze.airbnb_apartments
GROUP BY HOST_ID"""
df_apartment_cnt = df_apart.select(col('host_id'),col('apartment_id'))\
    .groupBy(col('host_id').alias("HOST_ID"))\
    .agg(count('apartment_id').alias("APARTMENT_CNT"))\
    .orderBy(col('host_id'))
display(df_apartment_cnt)

In [0]:
%python
"""
3.Retrieve all hosts who are female.
SELECT COUNT(host_id) AS `HOST_CNT` FROM ska_catalog.bronze.airbnb_hosts
WHERE gender LIKE 'F'
"""
df_female = df_host.where(col('gender').like('F%')) \
    .agg(count('host_id').alias('HOST_CNT'))
display(df_female)

In [0]:
-- 4.Show apartment details where number of bedrooms is greater than 1.

SELECT apartment_id, apartment_type, n_bedrooms, n_beds, country, city
FROM ska_catalog.bronze.airbnb_apartments
WHERE n_bedrooms > 1;

In [0]:
-- 5.Get the nationality and age of host with host_id = 3.
SELECT host_id,nationality,age FROM ska_catalog.bronze.airbnb_hosts
WHERE host_id = 3

In [0]:
-- List all apartments along with the host's nationality and age.
SELECT a.apartment_id, a.apartment_type, h.host_id, h.nationality, h.age 
FROM ska_catalog.bronze.airbnb_hosts h
JOIN ska_catalog.bronze.airbnb_apartments a
ON h.host_id = a.host_id;

In [0]:
-- Find hosts who own apartments in more than one city.
SELECT h.host_id, h.nationality, COUNT(DISTINCT a.city) AS city_count
FROM ska_catalog.bronze.airbnb_hosts h
JOIN ska_catalog.bronze.airbnb_apartments a
ON h.host_id = a.host_id
GROUP BY h.host_id, h.nationality
HAVING COUNT(DISTINCT a.city) > 1;

In [0]:
%python

df_city_count = df_host.join(df_apart, df_host.host_id == df_apart.host_id) \
    .groupBy(df_host.host_id, df_host.nationality) \
    .agg(countDistinct(df_apart.city).alias("city_count")) \
    .where(col("city_count") > 1)

display(df_city_count)

In [0]:
-- Get the average age of hosts by gender.
SELECT gender AS `GENDER`, ROUND(AVG(age)) AS `AVG_AGE`
FROM ska_catalog.bronze.airbnb_hosts
GROUP BY gender

In [0]:
-- List all apartment types and their count per country
SELECT country, apartment_type, COUNT(apartment_id) AS APARTMENT_CNT
FROM ska_catalog.bronze.airbnb_apartments
GROUP BY country, apartment_type

In [0]:
-- Find the cities where hosts younger than 30 have apartments.

SELECT DISTINCT a.city
FROM ska_catalog.bronze.airbnb_apartments a
JOIN ska_catalog.bronze.airbnb_hosts h
ON a.host_id = h.host_id
WHERE h.age < 30;

In [0]:
-- Find the host(s) who own the apartment with the maximum number of beds.
SELECT h.host_id, a.apartment_id, a.n_beds
FROM ska_catalog.bronze.airbnb_apartments a
JOIN ska_catalog.bronze.airbnb_hosts h
ON a.host_id = h.host_id
WHERE a.n_beds = (
  SELECT MAX(n_beds) FROM ska_catalog.bronze.airbnb_apartments
)


In [0]:
SELECT h.host_id, h.nationality, SUM(a.n_beds) AS total_beds,
       DENSE_RANK() OVER (ORDER BY SUM(a.n_beds) DESC) AS bed_rank
FROM ska_catalog.bronze.airbnb_hosts h
JOIN ska_catalog.bronze.airbnb_apartments a
ON h.host_id = a.host_id
GROUP BY h.host_id, h.nationality
ORDER BY total_beds DESC;